In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from statsmodels.graphics.tsaplots import plot_acf
import os

In [4]:
import pandas as pd

# Full path to CSV file
glucose_path = r"D:\MIU\4.1\Graduation\T1DiabetesGranada\T1DiabetesGranada\Glucose_measurements.csv"

# Read CSVs
glucose_df = pd.read_csv(glucose_path)

# Preprocessing


### Exploratory Data Analysis

In [3]:
print("Glucose shape:", glucose_df.shape)
print("Columns:", glucose_df.columns.tolist())
print("Unique patients:", glucose_df["Patient_ID"].nunique())

Glucose shape: (22671708, 4)
Columns: ['Patient_ID', 'Measurement_date', 'Measurement_time', 'Measurement']
Unique patients: 736


In [4]:
# ---- Combine date & time (REQUIRED) ----
glucose_df["Measurement_Datetime"] = pd.to_datetime(
    glucose_df["Measurement_date"] + " " + glucose_df["Measurement_time"],
    errors="coerce"
)

# Remove bad timestamps
glucose_df = glucose_df.dropna(subset=["Measurement_Datetime"])

In [5]:
# ---- Sort correctly ----
glucose_df = glucose_df.sort_values(
    ["Patient_ID", "Measurement_Datetime"]
).reset_index(drop=True)

In [6]:
# ---- Time difference per patient----
glucose_df["time_diff_min"] = (
    glucose_df.groupby("Patient_ID")["Measurement_Datetime"]
    .diff()
    .dt.total_seconds() / 60
)

In [7]:
# ---- Basic sanity checks you actually use later ----
print("\nMissing values:\n", glucose_df.isna().sum())
print("Duplicate rows:", glucose_df.duplicated().sum())
print("Measurement stats:\n", glucose_df["Measurement"].describe())
print("Time diff stats:\n", glucose_df["time_diff_min"].describe())



Missing values:
 Patient_ID                0
Measurement_date          0
Measurement_time          0
Measurement               0
Measurement_Datetime      0
time_diff_min           736
dtype: int64
Duplicate rows: 0
Measurement stats:
 count    2.267171e+07
mean     1.647837e+02
std      7.157474e+01
min      4.000000e+01
25%      1.120000e+02
50%      1.530000e+02
75%      2.060000e+02
max      5.000000e+02
Name: Measurement, dtype: float64
Time diff stats:
 count    2.267097e+07
mean     2.073637e+01
std      1.206392e+03
min      0.000000e+00
25%      1.500000e+01
50%      1.500000e+01
75%      1.500000e+01
max      1.516981e+06
Name: time_diff_min, dtype: float64


### Data Cleaning

In [8]:
glucose_df = glucose_df.drop_duplicates(
    subset=["Patient_ID", "Measurement_Datetime"]
)

In [9]:
dups_after = glucose_df.duplicated(
    subset=["Patient_ID", "Measurement_Datetime"]
).sum()

print("Duplicate (Patient + Datetime) AFTER cleaning:", dups_after)

Duplicate (Patient + Datetime) AFTER cleaning: 0


In [10]:
glucose_df = glucose_df[
    (glucose_df["Measurement"] >= 40) &
    (glucose_df["Measurement"] <= 400)
]

In [11]:
print("Measurement min/max AFTER cleaning:",
      glucose_df["Measurement"].min(),
      glucose_df["Measurement"].max())

Measurement min/max AFTER cleaning: 40 400


In [12]:
# recompute
glucose_df["time_diff_min"] = (
    glucose_df.groupby("Patient_ID")["Measurement_Datetime"]
    .diff()
    .dt.total_seconds() / 60
)
print(glucose_df["time_diff_min"].describe())

count    2.248355e+07
mean     2.090915e+01
std      1.211471e+03
min      1.000000e+00
25%      1.500000e+01
50%      1.500000e+01
75%      1.500000e+01
max      1.516981e+06
Name: time_diff_min, dtype: float64


In [13]:
# Remove huge gaps
glucose_df = glucose_df[
    (glucose_df["time_diff_min"].isna()) |
    (glucose_df["time_diff_min"] <= 30)
]

In [14]:
glucose_df = glucose_df.sort_values(
    ["Patient_ID", "Measurement_Datetime"]
).reset_index(drop=True)

big_gaps = (glucose_df["time_diff_min"] > 30).sum()
print("Gaps >30 min AFTER cleaning:", big_gaps)

zero_gaps = (glucose_df["time_diff_min"] == 0).sum()
print("Zero gaps AFTER cleaning:", zero_gaps)


Gaps >30 min AFTER cleaning: 0
Zero gaps AFTER cleaning: 0


In [15]:
print("Final cleaned shape:", glucose_df.shape)
print("Unique patients:", glucose_df["Patient_ID"].nunique())

Final cleaned shape: (22324151, 6)
Unique patients: 736


In [16]:
glucose_df.head()

,Patient_ID,Measurement_date,Measurement_time,Measurement,Measurement_Datetime,time_diff_min
0,LIB193263,2020-06-09,19:08:00,99,2020-06-09 19:08:00,NaN
1,LIB193263,2020-06-09,19:23:00,92,2020-06-09 19:23:00,15.0
2,LIB193263,2020-06-09,19:38:00,86,2020-06-09 19:38:00,15.0
3,LIB193263,2020-06-09,19:53:00,85,2020-06-09 19:53:00,15.0
4,LIB193263,2020-06-09,20:08:00,85,2020-06-09 20:08:00,15.0


In [20]:
WINDOW_SIZE = 24
HORIZON = 1
STRIDE = 6
BATCH_SIZE = 30  # number of patients per batch
LAST_DAYS = 90
SAVE_PATH = "windowed_data"
os.makedirs(SAVE_PATH, exist_ok=True)

### Create windows per patient 

In [21]:
def create_windows(df, window, horizon, stride):
    values = df['Measurement'].values.astype(np.float32)
    max_start = len(values) - window - horizon + 1
    if max_start <= 0:
        return None, None

    n_windows = (max_start + stride - 1) // stride
    X = np.empty((n_windows, window), dtype=np.float32)
    y = np.empty((n_windows, horizon), dtype=np.float32)

    idx = 0
    for start in range(0, max_start, stride):
        end = start + window
        X[idx] = values[start:end]
        y[idx] = values[end:end + horizon]
        idx += 1

    return X, y

# Last 90 days filtering
glucose_df['days_from_end'] = glucose_df.groupby('Patient_ID')['Measurement_Datetime'].transform(
    lambda x: (x.max() - x).dt.days
)
reduced_df = glucose_df[glucose_df['days_from_end'] <= LAST_DAYS].copy()

print(f"Rows after last {LAST_DAYS} days filter:", len(reduced_df))

# Batch processing by patients
unique_patients = reduced_df['Patient_ID'].unique()
batch_counter = 0

for start_idx in range(0, len(unique_patients), BATCH_SIZE):
    batch_ids = unique_patients[start_idx:start_idx + BATCH_SIZE]
    batch_df = reduced_df[reduced_df['Patient_ID'].isin(batch_ids)].copy()

    X_all, y_all = [], []

    for pid, pdf in batch_df.groupby('Patient_ID'):
        pdf = pdf.sort_values('Measurement_Datetime')
        Xp, yp = create_windows(pdf, WINDOW_SIZE, HORIZON, STRIDE)
        if Xp is not None:
            X_all.append(Xp)
            y_all.append(yp)

    if X_all:
        X_batch = np.concatenate(X_all)
        y_batch = np.concatenate(y_all)
        np.save(os.path.join(SAVE_PATH, f"X_batch_{batch_counter}.npy"), X_batch)
        np.save(os.path.join(SAVE_PATH, f"y_batch_{batch_counter}.npy"), y_batch)
        print(f"✓ Batch {batch_counter} saved: X {X_batch.shape}, y {y_batch.shape}")
        batch_counter += 1

    # free memory
    del X_all, y_all, X_batch, y_batch, batch_df
    import gc; gc.collect()


Rows after last 90 days filter: 4898405
✓ Batch 0 saved: X (31569, 24), y (31569, 1)
✓ Batch 1 saved: X (34540, 24), y (34540, 1)
✓ Batch 2 saved: X (32643, 24), y (32643, 1)
✓ Batch 3 saved: X (33741, 24), y (33741, 1)
✓ Batch 4 saved: X (31931, 24), y (31931, 1)
✓ Batch 5 saved: X (34827, 24), y (34827, 1)
✓ Batch 6 saved: X (33227, 24), y (33227, 1)
✓ Batch 7 saved: X (37903, 24), y (37903, 1)
✓ Batch 8 saved: X (36500, 24), y (36500, 1)
✓ Batch 9 saved: X (31036, 24), y (31036, 1)
✓ Batch 10 saved: X (35231, 24), y (35231, 1)
✓ Batch 11 saved: X (34143, 24), y (34143, 1)
✓ Batch 12 saved: X (32723, 24), y (32723, 1)
✓ Batch 13 saved: X (35899, 24), y (35899, 1)
✓ Batch 14 saved: X (33178, 24), y (33178, 1)
✓ Batch 15 saved: X (32608, 24), y (32608, 1)
✓ Batch 16 saved: X (31317, 24), y (31317, 1)
✓ Batch 17 saved: X (30196, 24), y (30196, 1)
✓ Batch 18 saved: X (32309, 24), y (32309, 1)
✓ Batch 19 saved: X (32161, 24), y (32161, 1)
✓ Batch 20 saved: X (34250, 24), y (34250, 1)
✓ Ba